In [15]:
import os
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from PIL import Image
from matplotlib import pyplot as plt
import logging
from tqdm import tqdm

In [2]:
logging.basicConfig(format="%(asctime)s - %(levelname)s: %(message)s", level=logging.INFO, datefmt="%I:%M:%S")

In [3]:
class Diffusion:
    '''
    1 - Setting up a noising schedule
    2 - Function for noising images
    3 - Sampling images
    '''
    
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=64, device="cuda"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device
        
        self.beta = self.prepare_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)
        
    def prepare_noise_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)
    
    def noise_images(self, x, t):
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1.0 - self.alpha_hat[t])[:, None, None, None]
        eps = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * eps, eps
    
    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.noise_steps, size=(n,))
    
    def sample(self, model, n):
        logging.info(f"Sampling {n} new images ....")
        model.eval()
        with torch.no_grad():
            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)
            for i in tqdm(reversed(range(1, self.noise_steps)), position=0):
                t = (torch.ones(n) * i).long().to(self.device)
                predicted_noise = model(x, t)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                if i>1:
                    noise = torch.randn_like(x)
                else:
                    noise = torch.zeros_like(x)
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha)/(torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        
        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
                            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
                            nn.GroupNorm(1, mid_channels),
                            nn.GELU(),
                            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
                            nn.GroupNorm(1, out_channels),
                            )
        
    def forward(self, x):
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        else:
            return self.double_conv(x)
        
class Down(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
                            nn.MaxPool2d(2),
                            DoubleConv(in_channels, in_channels, residual=True),
                            DoubleConv(in_channels, out_channels),
                            )
        
        self.emb_layer = nn.Sequential(
                            nn.SiLU(),
                            nn.Linear(emb_dim, out_channels),
                            )
        
    def forward(self, x, t):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb
    
class Up(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
                    DoubleConv(in_channels, in_channels, residual=True),
                    DoubleConv(in_channels, out_channels, in_channels//2),
                    )
        self.emb_layer = nn.Sequential(
                            nn.SiLU(),
                            nn.Linear(emb_dim, out_channels),
                            )
        
    def forward(self, x, skip_x, t):
        x = self.up(x)
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb
    
class SelfAttention(nn.Module):
    def __init__(self, channels, size):
        super().__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
                        nn.LayerNorm([channels]),
                        nn.Linear(channels, channels),
                        nn.GELU(),
                        nn.Linear(channels, channels),
                        )
        
    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)

In [5]:
class UNet(nn.Module):
    def __init__(self, c_in=3, c_out=3, time_dim=256, device="cuda"):
        super().__init__()
        self.device = device
        self.time_dim = time_dim
        self.inc = DoubleConv(in_channels=c_in, out_channels=64)
        self.down1 = Down(in_channels=64, out_channels=128)
        self.sa1 = SelfAttention(channels=128, size=32)
        self.down2 = Down(in_channels=128, out_channels=256)
        self.sa2 = SelfAttention(channels=256, size=16)
        self.down3 = Down(in_channels=256, out_channels=256)
        self.sa3 = SelfAttention(channels=256, size=8)
        
        self.bot1 = DoubleConv(in_channels=256, out_channels=512)
        self.bot2 = DoubleConv(in_channels=512, out_channels=512)
        self.bot3 = DoubleConv(in_channels=512, out_channels=256)
        
        self.up1 = Up(in_channels=512, out_channels=128)
        self.sa4 = SelfAttention(channels=128, size=16)
        self.up2 = Up(in_channels=256, out_channels=64)
        self.sa5 = SelfAttention(channels=64, size=32)
        self.up3 = Up(in_channels=128, out_channels=64)
        self.sa6 = SelfAttention(channels=64, size=64)
        self.outc = nn.Conv2d(64, c_out, kernel_size=1)
        
    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2, device=self.device).float() / channels))
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a ,pos_enc_b], dim=-1)
        return pos_enc
    
    def forward(self, x, t):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)
        
        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x2 = self.sa1(x2)
        x3 = self.down2(x2, t)
        x3 = self.sa2(x3)
        x4 = self.down3(x3, t)
        x4 = self.sa3(x4)
        
        x4 = self.bot1(x4)
        x4 = self.bot2(x4)
        x4 = self.bot3(x4)
        
        x = self.up1(x4, x3, t)
        x = self.sa4(x)
        x = self.up2(x, x2, t)
        x = self.sa5(x)
        x = self.up3(x, x1, t)
        x = self.sa6(x)
        output = self.outc(x)
        return output

In [6]:
def plot_images(images):
    plt.figure(figsize=(32,32))
    plt.imshow(torch.cat([
        torch.cat([i for i in images.cpu()], dim=-1)
    ], dim=-2).permute(1, 2, 0).cpu())
    plt.show()

In [7]:
def save_images(images, path, **kwargs):
    grid = torchvision.utils.make_grid(images, **kwargs)
    ndarr = grid.permute(1,2,0).to("cpu").numpy()
    im = Image.fromarray(ndarr)
    im.save(path)

In [8]:
def get_data(args):
    transforms = T.Compose([
        T.Resize(80),
        T.RandomResizedCrop(args.image_size, scale=(0.8, 1.0)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    dataset = ImageFolder(args.dataset_path, transform=transforms)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True)
    return dataloader

In [9]:
def setup_logging(run_name):
    os.makedirs("models", exist_ok=True)
    os.makedirs("results", exist_ok=True)
    os.makedirs(os.path.join("models", run_name), exist_ok=True)
    os.makedirs(os.path.join("results", run_name), exist_ok=True)

In [10]:
def train(args):
    setup_logging(args.run_name)
    device = args.device
    dataloader = get_data(args)
    model = UNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    mse = nn.MSELoss()
    diffusion = Diffusion(img_size=args.image_size, device=device)
    logger = SummaryWriter(os.path.join("runs", args.run_name))
    l = len(dataloader)
    
    for epoch in range(args.epochs):
        logging.info(f"Starting epoch {epoch}:")
        pbar = tqdm(dataloader)
        for i, (images, _) in enumerate(pbar):
            images = images.to(device)
            t = diffusion.sample_timesteps(images.shape[0]).to(device)
            x_t, noise = diffusion.noise_images(images, t)
            predicted_noise = model(x_t, t)
            loss = mse(noise, predicted_noise)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            pbar.set_postfix(MSE=loss.item())
            logger.add_scalar("MSE", loss.item(), global_step=epoch*l + i)
        
        sampled_images = diffusion.sample(model, n=images.shape[0])
        save_images(sampled_images, os.path.join("results", args.run_name, f"{epoch}.jpg"))
        torch.save(model.state_dict()), os.path.join("models", args.run_name, f"ckpt.pt")

In [11]:
def launch():
    import argparse
    parser = argparse.ArgumentParser()
    args = parser.parse_args()
    args.run_name = "DDPM_Unconditional"
    args.epochs = 3
    args.batch_size = 12
    args.image_size = 64
    args.dataset_path = r"C:\Users\subir\OneDrive\Desktop\Face Recognition\Diffusion Model\Data"
    args.device = "cuda"
    args.lr = 3e-4
    train(args)

In [12]:
if __name__ == "__main__":
    launch()

usage: ipykernel_launcher.py [-h]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\subir\AppData\Roaming\jupyter\runtime\kernel-18c109cd-2eae-46f1-b6ce-78a51ce8d0db.json


SystemExit: 2

C:\Users\subir\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3377: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [13]:
%tb

SystemExit: 2

In [ ]:
torch.randn_like?

In [ ]:
torch.randn(5)

In [ ]:
torch.linspace?

In [ ]:
a = 1. - torch.linspace(25, 37, 13)
a

In [ ]:
torch.cumprod?

In [ ]:
b = torch.cumprod(a, dim=0)
b

In [ ]:
torch.sqrt(b).shape

In [ ]:
torch.sqrt(b)[:].shape

In [ ]:
torch.sqrt(b)[:, None].shape

In [ ]:
torch.sqrt(b)[:, None, None, None].shape

In [ ]:
torch.randint(1, 50, size=(3,))

In [ ]:
for i in reversed(range(1, 10)):
    print(i)

In [ ]:
torch.ones(5) * 3

In [ ]:
a = torch.randn((1, 128, 32, 32))
print(a.shape)
a.view(-1, 128, 32 * 32).swapaxes(1, 2).shape

In [19]:
np.random.random()

0.807759753000116

In [20]:
torch.lerp?